# Tarea 1 – Parte 1: Recolección y limpieza de noticias

**Pregunta guía:** ¿se pueden convertir titulares de un diario peruano en un clasificador de noticias?

| Configuración | Valor |
| :--- | :--- |
| Fecha de descarga | **2026-10-18** (hora de Lima) |
| Opción elegida | **B – El Comercio** (350–400 noticias, ~90–100 por categoría) |
| URL base | `https://elcomercio.pe` |
| `robots.txt` | `https://elcomercio.pe/robots.txt` |
| Índices de sitemaps (declarados en `robots.txt`) | `https://elcomercio.pe/sitemap/{seccion}/?outputType=xml` |
| Sitemaps usados (URL y fecha de cada nota) | `https://elcomercio.pe/sitemap/web/{seccion}/?outputType=xml` con `seccion` = `politica`, `economia`, `deporte-total`, `tecnologia` |
| Titular, bajada y fecha | HTML de cada nota (`requests` + `BeautifulSoup`) |
| Categorías finales | `politica`, `economia`, `deportes`, `ciencia_tecnologia` |

**¿Por qué sitemaps + HTML y no solo RSS?**

- El Comercio tiene un RSS oficial por sección (`/arc/outboundfeeds/google-news-feed/category/{seccion}/`), pero solo cubre las últimas ~48 horas. El 18/10 traía 20 notas de política, 13 de economía, 86 de deportes y 15 de tecnología, así que no alcanza para ~90 por categoría.
- Los sitemaps de sección que publica `robots.txt` listan las 100 notas más recientes de cada sección, con su URL y su fecha. Con esa lista se visita cada nota **una sola vez** para leer el titular, la bajada y la fecha de publicación.
- Se descartó La República (opciones A y C) porque sus RSS y sitemaps por sección traen solo entre 6 y 21 notas, y su `robots.txt` prohíbe paginar las secciones (`Disallow: /*/pag/`).

**Buenas prácticas de scraping:** se revisa `robots.txt` antes de descargar, se usa un `User-Agent` descriptivo con enlace de contacto, hay una pausa de 1.5 s entre solicitudes y como máximo 1 reintento por URL. Todo lo descargado se respalda en `datos/raw/`; al reejecutar el notebook no se vuelve a pedir al servidor lo que ya está guardado.

In [1]:
import json
import time
import urllib.robotparser
from datetime import date
from pathlib import Path
from urllib.parse import urlparse

import pandas as pd
import plotly.express as px
import plotly.io as pio
import requests
import truststore
from bs4 import BeautifulSoup

# Valida HTTPS con los certificados del sistema operativo. En Windows, el antivirus
# intercepta HTTPS con su propio certificado raíz, que el certifi de requests no conoce.
truststore.inject_into_ssl()

# Plotly: interactivo en Jupyter/VS Code + copia PNG para que GitHub muestre los gráficos
pio.renderers.default = "plotly_mimetype+png"
pio.renderers["png"].width = 950
pio.renderers["png"].height = 500
pd.set_option("display.max_colwidth", 110)

OPCION = "B"
MEDIO = "El Comercio"
BASE = "https://elcomercio.pe"
USER_AGENT = ("Mozilla/5.0 (compatible; TareaDiplomadoPython/1.0; uso academico; "
              "+https://github.com/IvettexMamani/diplomado-python-2026)")
PAUSA_SEG = 1.5      # entre 1 y 2 s entre solicitudes
MAX_REINTENTOS = 1   # como máximo 1 reintento por URL

# Secciones del diario cuyo sitemap se recorre
SITEMAPS = ["politica", "economia", "deporte-total", "tecnologia"]
ETIQUETAS = ["politica", "economia", "deportes", "ciencia_tecnologia"]

DATOS = Path("datos")
RAW = DATOS / "raw"
(RAW / "sitemaps").mkdir(parents=True, exist_ok=True)

print("Fecha de ejecución:", date.today().isoformat())
print(f"Opción {OPCION}: {MEDIO} ({BASE})")

Fecha de ejecución: 2026-10-18
Opción B: El Comercio (https://elcomercio.pe)


## 1. Revisión de `robots.txt`
Se descarga `robots.txt` (y se guarda una copia en `datos/raw/`), se leen las reglas que aplican a cualquier robot (`User-agent: *`) y se comprueba con `urllib.robotparser` que el `User-Agent` del proyecto puede leer los sitemaps y las notas.

In [2]:
sesion = requests.Session()
sesion.headers.update({"User-Agent": USER_AGENT, "Accept-Language": "es-PE,es;q=0.9"})
_ultima = [0.0]  # momento de la última solicitud


def descargar(url):
    # GET respetuoso: pausa mínima entre solicitudes y como máximo 1 reintento.
    r = None
    for intento in range(1 + MAX_REINTENTOS):
        espera = PAUSA_SEG - (time.time() - _ultima[0])
        if espera > 0:
            time.sleep(espera)
        try:
            r = sesion.get(url, timeout=25)
        except requests.RequestException as e:
            print(f"  error de red ({type(e).__name__}) en {url}")
            r = None
        _ultima[0] = time.time()
        if r is not None and r.status_code in (200, 404, 410):
            return r  # éxito, o la página no existe: no tiene sentido reintentar
    return r


archivo_robots = RAW / "robots_elcomercio.txt"
if archivo_robots.exists():
    texto_robots = archivo_robots.read_text(encoding="utf-8")
    print("robots.txt leído del respaldo en datos/raw/")
else:
    r = descargar(f"{BASE}/robots.txt")
    r.raise_for_status()
    texto_robots = r.text
    archivo_robots.write_text(texto_robots, encoding="utf-8")
    print("robots.txt descargado y guardado en datos/raw/")

rp = urllib.robotparser.RobotFileParser()
rp.parse(texto_robots.splitlines())

# Reglas del grupo "User-agent: *", que es el que aplica a nuestro User-Agent
reglas, en_grupo = [], False
for linea in texto_robots.splitlines():
    linea = linea.strip()
    if linea.lower().startswith("user-agent:"):
        en_grupo = linea.split(":", 1)[1].strip() == "*"
    elif en_grupo and linea.lower().startswith(("disallow:", "allow:")):
        reglas.append(linea)
sitemaps_declarados = [linea.split(":", 1)[1].strip() for linea in texto_robots.splitlines()
                       if linea.lower().startswith("sitemap:")]

print("Reglas para 'User-agent: *':", reglas)
print("Sitemaps declarados en robots.txt:", len(sitemaps_declarados))
print("Índices de las 4 secciones declarados:",
      all(f"{BASE}/sitemap/{s}/?outputType=xml" in sitemaps_declarados for s in SITEMAPS))

robots.txt descargado y guardado en datos/raw/
Reglas para 'User-agent: *': ['Disallow: /catalogo/', 'Disallow: /buscar/', 'Disallow: /archivo/todas/19*', 'Disallow: /archivo/todas/200*']
Sitemaps declarados en robots.txt: 44
Índices de las 4 secciones declarados: True


In [3]:
pruebas = {
    "Índice de sitemaps de política": f"{BASE}/sitemap/politica/?outputType=xml",
    "Sitemap web de deportes": f"{BASE}/sitemap/web/deporte-total/?outputType=xml",
    "Nota de economía (ejemplo)": f"{BASE}/economia/mercados/ejemplo-noticia/",
    "Nota de tecnología (ejemplo)": f"{BASE}/tecnologia/actualidad/ejemplo-noticia/",
    "Buscador": f"{BASE}/buscar/ejemplo/",
    "Catálogo": f"{BASE}/catalogo/",
}
pd.DataFrame({"recurso": list(pruebas), "url": list(pruebas.values()),
              "permitido": [rp.can_fetch(USER_AGENT, u) for u in pruebas.values()]})

,recurso,url,permitido
0,Índice de sitemaps de política,https://elcomercio.pe/sitemap/politica/?outputType=xml,True
1,Sitemap web de deportes,https://elcomercio.pe/sitemap/web/deporte-total/?outputType=xml,True
2,Nota de economía (ejemplo),https://elcomercio.pe/economia/mercados/ejemplo-noticia/,True
3,Nota de tecnología (ejemplo),https://elcomercio.pe/tecnologia/actualidad/ejemplo-noticia/,True
4,Buscador,https://elcomercio.pe/buscar/ejemplo/,False
5,Catálogo,https://elcomercio.pe/catalogo/,False


**Conclusión:** para cualquier robot, `robots.txt` solo prohíbe `/catalogo/`, `/buscar/` y el archivo histórico (`/archivo/todas/19*`, `/archivo/todas/200*`). El bloqueo total (`Disallow: /`) aplica solo a una lista de descargadores masivos (HTTrack, wget, WebZIP, etc.), y nuestro `User-Agent` no está en ella. Los sitemaps de sección y las páginas de las notas están permitidos.

**Fuente definida:** sitemaps oficiales por sección + HTML de cada nota.